# Feature Scaling or Normalization

_Feature Engineering (Zheng & Casari)_

**Put features that live on wildly different scales onto a common footing — so distance- and gradient-based models stop being bullied by the big-numbered column.**

A feature's units are arbitrary. Whether you record a length in millimeters or kilometers does not change the underlying information, yet it changes the raw number by a factor of a million. Many models cannot tell the difference between "this feature carries more signal" and "this feature just has bigger numbers".

       Scaling removes the units. It rewrites every feature into a comparable range so that the model judges features by their information, not their magnitude. Crucially, all three methods here are affine per-feature (min-max, standardization) or per-row rescaling (L2) — they move and stretch points but never bend the cloud. The shape of each feature's histogram is preserved; only its location and spread change.

---

This notebook is a practice scaffold for the **Feature Scaling or Normalization** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import pandas as pd
from sklearn import preprocessing

# Online News Popularity dataset (UCI). Get it from the book's repo:
#   github.com/alicezheng/feature-engineering-book
df = pd.read_csv('OnlineNewsPopularity.csv')
df.columns = [c.strip() for c in df.columns]   # the CSV has leading spaces

# The book scales n_tokens_content: the number of words in an article.
# preprocessing.minmax_scale expects a 2-D array, so pass a single-column DataFrame.

# 1. MIN-MAX SCALING -> every value mapped into [0, 1].
df['minmax'] = preprocessing.minmax_scale(df[['n_tokens_content']])

# 2. STANDARDIZATION (variance scaling / z-score) -> mean 0, variance 1.
df['standardized'] = (
    preprocessing.StandardScaler().fit_transform(df[['n_tokens_content']])
)

# 3. L2 NORMALIZATION -> divide each ROW vector by its L2 norm (unit sphere).
#    (Per-row: here on a single column it just maps each value to +/-1, but on a
#     full feature matrix it makes every example's vector have length 1.)
df['l2_normalized'] = preprocessing.normalize(
    df[['n_tokens_content']], norm='l2', axis=0
)

print(df[['n_tokens_content', 'minmax', 'standardized', 'l2_normalized']].head())

# Scaling changes the RANGE, not the SHAPE: plot to confirm the histograms
# have identical shape, only different x-axis units.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 1, figsize=(6, 8))
df['n_tokens_content'].hist(ax=axes[0], bins=100)
axes[0].set_title('Original n_tokens_content')
df['minmax'].hist(ax=axes[1], bins=100)
axes[1].set_title('Min-max scaled to [0, 1]')
df['standardized'].hist(ax=axes[2], bins=100)
axes[2].set_title('Standardized (mean 0, var 1)')
plt.tight_layout()
plt.show()

## Visualize the data & results

_Two real features on wildly different scales: wine 'proline' (~hundreds) vs 'hue' (~1). What do their min / mean / max look like BEFORE standardization, and how does standardization put them on a common footing?_

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

d = load_wine()
names = list(d.feature_names)
pi, hi = names.index('proline'), names.index('hue')
proline = d.data[:, pi]   # amino acid, ~hundreds
hue     = d.data[:, hi]   # color ratio, ~1

def mmm(a):  # min, mean, max
    return round(a.min(), 3), round(a.mean(), 3), round(a.max(), 3)

print('BEFORE proline min/mean/max:', mmm(proline))  # (278.0, 746.893, 1680.0)
print('BEFORE hue     min/mean/max:', mmm(hue))       # (0.48, 0.957, 1.71)

# Standardize each feature: subtract mean, divide by std -> mean 0, variance 1.
Z = StandardScaler().fit_transform(np.c_[proline, hue])
print('AFTER  proline min/mean/max:', mmm(Z[:, 0]))   # (-1.493, -0.0, 2.971)
print('AFTER  hue     min/mean/max:', mmm(Z[:, 1]))   # (-2.095, 0.0, 3.302)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You feed a k-nearest-neighbours classifier two features: annual income (tens of thousands) and number of children (0-5). Without scaling, the model is essentially ignoring the number of children. Explain why, and fix it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the Euclidean distance between two people across the two raw features. — _Distance squared is (income difference)^2 + (children difference)^2; income differences are in the thousands, children differences at most 5._
- Observe that the income term dwarfs the children term by a factor of millions once squared. — _k-NN finds 'nearest' points almost entirely by income, so number of children barely affects who counts as a neighbor._
- Standardize both features (subtract mean, divide by standard deviation), fit on the training set only. — _Now both features have variance 1, so they contribute comparably to the distance and the model can use both._

**Answer:** k-NN uses Euclidean distance, and income's raw magnitude (tens of thousands) overwhelms children (0-5), so 'nearest' is decided almost entirely by income. Standardize each feature to mean 0, variance 1 (fit the scaler on training data only, inside a Pipeline). Then both features contribute comparably and the model uses number of children too.

</details>

**Problem 2.** A teammate runs StandardScaler().fit_transform(X) on the full dataset, THEN splits into train and test and reports a cross-validation AUC. Why is the reported score optimistic, and what is the correct protocol?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify that the scaler's mean and standard deviation were computed using the test rows too. — _Information about the held-out rows (their statistics) has flowed into the transform applied to the training rows._
- Name this as data leakage: the validation rows are no longer truly unseen. — _The model gets a peek at the held-out distribution, so the validation score overstates real-world performance._
- Move the scaler inside a Pipeline and fit it within each cross-validation fold, on the training fold only. — _A Pipeline re-fits the scaler on each fold's training data, so the held-out fold's statistics never leak in._

**Answer:** Fitting the scaler on all of X lets the test rows' mean and standard deviation leak into the training transform — that is leakage, and it inflates the AUC. Correct protocol: put StandardScaler inside a Pipeline so it is fit on the training fold only within each cross-validation split. The honest score will usually be a bit lower.

</details>

**Problem 3.** You have a right-skewed feature (a long tail of large values). A colleague says "just standardize it and the skew will go away." Is that right? What actually removes the skew, and in what order relative to scaling?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that standardization is an affine map: subtract a constant, divide by a constant. — _Affine maps only shift and stretch the axis; they cannot bend the distribution or shorten a tail._
- Conclude the histogram's shape (the skew) is unchanged by standardization — only its center and spread move. — _A right-skewed feature is still right-skewed by the same amount after standardizing._
- Apply a log or power transform FIRST to compress the tail, THEN standardize the result for the model. — _The non-linear log/power transform reshapes the distribution; scaling afterward puts it on a comparable footing._

**Answer:** No — standardization is an affine (shift-and-scale) map, so it leaves the distribution's shape, including the skew, unchanged. To remove skew you need a non-linear log or power transform, applied BEFORE scaling. Then standardize the transformed feature for the model.

</details>

**Problem 4.** You are about to put a StandardScaler in front of a gradient-boosted tree model. A reviewer says it is pointless. Are they right? What kind of model WOULD need the scaler here?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how a tree splits: it asks 'is feature x below a threshold t?'. — _The split depends only on the ordering of values, not their absolute magnitude._
- Note that any increasing rescaling maps the threshold along with the values, so the same rows fall on each side. — _Monotone rescaling leaves every split — and therefore the whole tree — identical, so scaling does nothing._
- Identify the models that DO need scaling: linear/SGD/neural-net and any distance-based model (k-NN, k-means, RBF-SVM). — _Those use gradients or distances, which are sensitive to feature magnitude, unlike threshold splits._

**Answer:** The reviewer is right: tree-based models split on thresholds, and any monotone rescaling moves the threshold with the values, leaving the splits unchanged — so scaling is wasted work for gradient-boosted trees. Scaling matters for gradient-based models (linear with regularization, SGD, neural nets) and distance-based models (k-NN, k-means, RBF-SVM).

</details>